<div align="center">

# Week 11: Mobile, Multimodal and Ubiquitous Interaction

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-11) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/Week11_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the screen of the phone with its grid of cells, the four main actions of the clinic app and their two layouts. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The phone of the clinic app: a screen of 68 by 148 mm with a grid of 3
# columns and 6 rows for the main actions (x from the left, y from the bottom)
PHONE = {'w': 68.0, 'h': 148.0, 'cols': 3, 'rows': 6}
CELL_X = [11.3, 34.0, 56.7]
CELL_Y = [12.3, 37.0, 61.7, 86.3, 111.0, 135.7]
ACTIONS = ['Book', 'My visits', 'Call', 'Directions']
LAYOUTS = {"current": {"Book": (2, 5), "My visits": (0, 5),
                        "Call": (1, 5), "Directions": (2, 4)},
           "stage 11": {"Book": (1, 1), "My visits": (1, 2),
                        "Call": (1, 0), "Directions": (2, 2)}}
print("Setup complete. Running in Colab:", IN_COLAB, "| actions:", ACTIONS,
      "| layouts:", list(LAYOUTS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 11.1": (lambda: 2 * math.hypot(40, 60), 1e-6),
    "Exercise 11.2": (lambda: (80 + 20 + 80) / 3, 1e-9),
    "Exercise 11.3": (lambda: -4.0, 1e-6),
    "Exercise A": (lambda: math.hypot(80 - 34.0, 30 - 61.7), 1e-6),
    "Exercise B": (lambda: 1 - 23.5 / (0.5 * math.sqrt(2 * 250 ** 2)), 1e-6),
    "Exercise C": (lambda: (26 - 1) / 9.4 * 60 / 5, 1e-6),
    "Exercise D": (lambda: 2 / 7, 1e-6),
    "Exercise E": (lambda: math.exp(-3.97 * 3 / 60), 1e-6),
    "Exercise F": (lambda: (0.82 / 0.18 * 1.48) / (1 + 0.82 / 0.18 * 1.48), 1e-6),
}

## Python step 11: Points and paths with NumPy

A finger on a touch screen leaves a path of points, and the recognizers of this week work on such paths. This step stores points as rows of a NumPy array &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-1)&#93;. It measures the length of a path, places points at equal distances along it and turns points around their centre. Section 2 builds a gesture recognizer from these pieces.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "LAYOUTS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Points as rows of an array

In [ ]:
# Python practice: a small example for learning the language
stroke = np.array([[10, 55], [38, 85], [90, 15]])   # a check mark: x, y
print(stroke.shape)            # 3 points with 2 coordinates each
print(stroke[:, 0])            # all x coordinates
print(stroke[1])               # the second point
print(stroke.mean(axis=0))     # the centroid: the mean x and the mean y

Each row of the array is one point, and each column one coordinate. `stroke[:, 0]` takes every row of column 0, the x coordinates, and `stroke[1]` takes row 1, the second point, because counting starts at 0. `mean(axis=0)` averages down the rows and returns one mean per column: The centroid of the stroke.

### Distances along a path

In [ ]:
# Python practice: a small example for learning the language
steps = np.diff(stroke, axis=0)           # from each point to the next
seg = np.hypot(steps[:, 0], steps[:, 1])    # length of each segment
print(steps)
print(seg.round(3), "| total length:", seg.sum().round(3))
print(np.cumsum(seg).round(3))              # length travelled so far

`np.diff` subtracts each row from the next one and so returns the steps of the path. `np.hypot` computes $\sqrt{dx^2 + dy^2}$ for every step at once, and the sum is the length of the stroke. `np.cumsum` adds the lengths up one by one: The distance travelled at each point.

### Points at equal distances with np.interp

In [ ]:
# Python practice: a small example for learning the language
dist = np.concatenate([[0], np.cumsum(seg)])   # distance at each point
targets = np.linspace(0, dist[-1], 5)          # 5 evenly spaced distances
xs = np.interp(targets, dist, stroke[:, 0])
ys = np.interp(targets, dist, stroke[:, 1])
print(np.column_stack([xs, ys]).round(1))

`np.linspace` returns five distances from the start to the end of the path. `np.interp` finds, for each distance, the place between the two points that surround it and interpolates x and y linearly. The five new points lie at equal distances along the stroke, whatever the speed of the finger was.

### Turning points with a rotation

In [ ]:
# Python practice: a small example for learning the language
theta = np.radians(90)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
centred = stroke - stroke.mean(axis=0)         # the centroid moves to (0, 0)
turned = centred @ R.T                         # every row turned by 90 degrees
print(turned.round(1))

A rotation by the angle $\theta$ multiplies each point by the matrix $R$. Subtracting the centroid first turns the stroke around its own centre. `@` multiplies matrices, and `R.T`, the transpose, is needed because the points are rows rather than columns.

**Quick check.** Why does `stroke.mean(axis=0)` return two numbers, while `stroke.mean()` returns only one?

<details><summary>Answer</summary>

With `axis=0` the mean runs down each column, so it gives the mean x and the mean y. Without an axis, all six numbers are averaged together.

</details>

### Exercises for Python step 11

**Exercise 11.1.** A caret runs through the points (10, 80), (50, 20) and (90, 80). Compute the length of the path with `np.diff` and `np.hypot` and store it in `length`.

In [ ]:
length = None   # your answer

check("Exercise 11.1", length)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
caret = np.array([[10, 80], [50, 20], [90, 80]])
steps = np.diff(caret, axis=0)
length = np.hypot(steps[:, 0], steps[:, 1]).sum()

check("Exercise 11.1", length)

**Exercise 11.2.** Store the y coordinate of the centroid of the same caret in `cy`.

In [ ]:
cy = None   # your answer

check("Exercise 11.2", cy)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
cy = np.array([[10, 80], [50, 20], [90, 80]]).mean(axis=0)[1]

check("Exercise 11.2", cy)

**Exercise 11.3.** Turn the point (3, 4) by 90 degrees around the origin with the matrix `R` and store the new x coordinate in `x_new`.

In [ ]:
x_new = None   # your answer

check("Exercise 11.3", x_new)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
x_new = (np.array([3, 4]) @ R.T)[0]

check("Exercise 11.3", x_new)

## 1. One hand: The reach of the thumb &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-2)&#93;

*Lecture: [Phones in the hand: Grips and the reach of the thumb](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#thumb-reach)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "turned" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A phone held in one hand is operated with the thumb of the same hand, and the thumb reaches only part of the screen. Python checks which buttons a thumb reaches and applies the check to the ten patients of Part A and to all patients of the clinic. It then compares the current layout with the layout of stage 11.

**From the formulas to the code.** The base of the thumb lies at $(x_p, y_p)$, 12 mm outside the edge of the screen at the grip height $g$. `thumb_distance(x, y, hand, g)` computes $d = \sqrt{(x - x_p)^2 + (y - y_p)^2}$, and `in_reach(x, y, hand, L, g)` tests $r L \le d \le L$ with $r = 0.32$. `cell_share(c, r)` gives the share of all patients who reach a cell, and `reach_share(layout)` averages it over the actions.

In [ ]:
PIVOT_OUT, R_MIN = 12.0, 0.32

def pivot(hand, g):
    """Base of the thumb for a phone held in the right or the left hand."""
    x = PHONE["w"] + PIVOT_OUT if hand == "R" else -PIVOT_OUT
    return x, g

def thumb_distance(x, y, hand, g):
    """Distance in mm from the base of the thumb to a point of the screen."""
    px, py = pivot(hand, g)
    return math.hypot(x - px, y - py)

def in_reach(x, y, hand, L, g):
    """True if the point lies in the functional area of the thumb."""
    return R_MIN * L <= thumb_distance(x, y, hand, g) <= L

for name, layout in LAYOUTS.items():
    c, r = layout["Book"]
    d = thumb_distance(CELL_X[c], CELL_Y[r], "R", 30)
    print(f"{name}: Book at ({CELL_X[c]}, {CELL_Y[r]}) mm | Ayşe: distance "
          f"{d:.2f} mm, in reach {in_reach(CELL_X[c], CELL_Y[r], 'R', 80, 30)}")

**Python note.** `pivot` returns two values at once as a tuple, and `px, py = pivot(...)` unpacks them. The test `R_MIN * L <= d <= L` chains two comparisons, as in mathematics.

In [ ]:
THUMB_PATIENTS = [   # name, age, hand, thumb length L, grip height g (mm)
    ("Ayşe", 34, "R", 80, 30),
    ("Mehmet", 45, "R", 90, 28),
    ("Elif", 28, "L", 81, 32),
    ("Hasan", 71, "R", 78, 22),
    ("Zeynep", 31, "L", 76, 30),
    ("Can", 24, "L", 84, 34),
    ("Selin", 38, "R", 79, 30),
    ("Emine", 74, "R", 72, 36),
    ("Burak", 19, "R", 88, 40),
    ("Fatma", 66, "L", 74, 26)]

for name, layout in LAYOUTS.items():
    served = []
    for who, age, hand, L, g in THUMB_PATIENTS:
        n_ok = sum(in_reach(CELL_X[c], CELL_Y[r], hand, L, g)
                   for c, r in layout.values())
        served.append(n_ok)
    print(f"{name}: {sum(served)} of {4 * len(served)} actions in reach |",
          ", ".join(f"{p[0]} {s}" for p, s in zip(THUMB_PATIENTS, served)))

**Python note.** `sum` over a sequence of `True` and `False` counts the `True` values, so `n_ok` is the number of actions that a patient reaches. `zip` pairs each patient with the count.

**Exercise A.** Ayşe holds the phone in her right hand with $g = 30$ mm, so the base of her thumb lies at (80, 30). Compute the distance to a button at (34.0, 61.7) mm and store it in `d_btn`. Is it in reach for $L = 80$ mm?

In [ ]:
d_btn = None   # your answer

check("Exercise A", d_btn)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d_btn = thumb_distance(34.0, 61.7, "R", 30)
print(in_reach(34.0, 61.7, "R", 80, 30))

check("Exercise A", d_btn)

In [ ]:
HANDS = {'L_mean': 82.0, 'L_sd': 7.0,
         'g_low': 20.0, 'g_high': 45.0, 'p_left': 0.25}

def simulate_hands(n=100_000, seed=RANDOM_STATE):
    """Thumb lengths, grip heights and the hand of many patients."""
    rng = np.random.default_rng(seed)
    L = rng.normal(HANDS["L_mean"], HANDS["L_sd"], n)
    g = rng.uniform(HANDS["g_low"], HANDS["g_high"], n)
    left = rng.random(n) < HANDS["p_left"]
    return L, g, left

L_all, g_all, left_all = simulate_hands()

def cell_share(c, r):
    """Share of the patients whose thumb reaches the centre of a cell."""
    px = np.where(left_all, -PIVOT_OUT, PHONE["w"] + PIVOT_OUT)
    d = np.hypot(CELL_X[c] - px, CELL_Y[r] - g_all)
    return float(((d >= R_MIN * L_all) & (d <= L_all)).mean())

def reach_share(layout):
    """Average share of the patients who reach the actions of a layout."""
    return float(np.mean([cell_share(c, r) for c, r in layout.values()]))

grid = np.array([[cell_share(c, r) for c in range(3)] for r in range(6)])
print("share who reach each cell, top row first:")
print(pd.DataFrame(grid[::-1].round(2), index=[f"row {r}" for r in range(5, -1, -1)],
                   columns=["left", "middle", "right"]))
for name, layout in LAYOUTS.items():
    print(f"{name}: " + ", ".join(f"{a} {cell_share(*cr):.3f}"
                                  for a, cr in layout.items())
          + f" | mean {reach_share(layout):.3f}")

**Python note.** `np.where(left_all, a, b)` takes `a` where a patient holds the phone in the left hand and `b` elsewhere. The operator `&` combines two arrays of truth values element by element, and the mean of the result is the share of patients for whom both conditions hold.

In [ ]:
# Figure: the reach of three thumbs and the share who reach each cell
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
a = ax[0]
a.add_patch(Rectangle((0, 0), PHONE["w"], PHONE["h"], fill=False, lw=2))
xs, ys = np.meshgrid(np.linspace(0, PHONE["w"], 137),
                     np.linspace(0, PHONE["h"], 297))
for (hand, L, g), col in zip([("R", 72, 36), ("R", 90, 28), ("L", 81, 32)],
                             ["#B53F2E", "#2E8B57", "#5A4BB5"]):
    ok = np.vectorize(lambda x, y: in_reach(x, y, hand, L, g))(xs, ys)
    a.contour(xs, ys, ok.astype(float), levels=[0.5], colors=col)
    a.plot([], [], color=col, label=f"{hand} hand, L = {L} mm")
a.set_xlim(-20, 90)
a.set_aspect("equal")
a.legend(fontsize=7, loc="upper right")
b = ax[1]
im = b.imshow(grid, origin="lower", cmap="Greens", vmin=0, vmax=1,
              extent=[0, PHONE["w"], 0, PHONE["h"]])
for r in range(6):
    for c in range(3):
        b.text(CELL_X[c], CELL_Y[r] - 8, f"{grid[r, c]:.2f}", ha="center",
               fontsize=8)
for (name, layout), m in zip(LAYOUTS.items(), ["s", "o"]):
    b.plot([CELL_X[c] for c, r in layout.values()],
           [CELL_Y[r] + 4 for c, r in layout.values()], m, mfc="none", ms=18,
           mew=2, label=name,
           mec="#B53F2E" if name == "current" else "#C98510")
b.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2)
plt.colorbar(im, ax=b, label="share of patients")
plt.tight_layout()
plt.show()

**Python note.** `np.meshgrid` makes a fine grid of points over the screen, and `contour` draws the border of the area in reach for each thumb. `imshow` paints the share of each cell, with `origin="lower"` so that row 0 is at the bottom, as on the phone.

**Paste your results.** Copy the results of Part A of the interactive lab and paste them between the triple quotes of `MY_LAYOUT` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_LAYOUT = """
"""
DEFAULT_LAYOUT = """
round,name,hand,L_mm,grip_mm,action,col,row,x_mm,y_mm,dist_mm,reached
1,Ayşe,R,80,30,"Book",2,5,56.7,135.7,108.2,0
1,Ayşe,R,80,30,"My visits",0,5,11.3,135.7,126.1,0
1,Ayşe,R,80,30,"Call",1,5,34.0,135.7,115.3,0
1,Ayşe,R,80,30,"Directions",2,4,56.7,111.0,84.3,0
1,Mehmet,R,90,28,"Book",2,5,56.7,135.7,110.2,0
1,Mehmet,R,90,28,"My visits",0,5,11.3,135.7,127.7,0
1,Mehmet,R,90,28,"Call",1,5,34.0,135.7,117.1,0
1,Mehmet,R,90,28,"Directions",2,4,56.7,111.0,86.2,1
1,Elif,L,81,32,"Book",2,5,56.7,135.7,124.4,0
1,Elif,L,81,32,"My visits",0,5,11.3,135.7,106.3,0
1,Elif,L,81,32,"Call",1,5,34.0,135.7,113.4,0
1,Elif,L,81,32,"Directions",2,4,56.7,111.0,104.7,0
1,Hasan,R,78,22,"Book",2,5,56.7,135.7,116.1,0
1,Hasan,R,78,22,"My visits",0,5,11.3,135.7,132.8,0
1,Hasan,R,78,22,"Call",1,5,34.0,135.7,122.7,0
1,Hasan,R,78,22,"Directions",2,4,56.7,111.0,92.0,0
1,Zeynep,L,76,30,"Book",2,5,56.7,135.7,126.1,0
1,Zeynep,L,76,30,"My visits",0,5,11.3,135.7,108.2,0
1,Zeynep,L,76,30,"Call",1,5,34.0,135.7,115.3,0
1,Zeynep,L,76,30,"Directions",2,4,56.7,111.0,106.2,0
1,Can,L,84,34,"Book",2,5,56.7,135.7,122.7,0
1,Can,L,84,34,"My visits",0,5,11.3,135.7,104.3,0
1,Can,L,84,34,"Call",1,5,34.0,135.7,111.6,0
1,Can,L,84,34,"Directions",2,4,56.7,111.0,103.2,0
1,Selin,R,79,30,"Book",2,5,56.7,135.7,108.2,0
1,Selin,R,79,30,"My visits",0,5,11.3,135.7,126.1,0
1,Selin,R,79,30,"Call",1,5,34.0,135.7,115.3,0
1,Selin,R,79,30,"Directions",2,4,56.7,111.0,84.3,0
1,Emine,R,72,36,"Book",2,5,56.7,135.7,102.4,0
1,Emine,R,72,36,"My visits",0,5,11.3,135.7,121.1,0
1,Emine,R,72,36,"Call",1,5,34.0,135.7,109.8,0
1,Emine,R,72,36,"Directions",2,4,56.7,111.0,78.5,0
1,Burak,R,88,40,"Book",2,5,56.7,135.7,98.5,0
1,Burak,R,88,40,"My visits",0,5,11.3,135.7,117.8,0
1,Burak,R,88,40,"Call",1,5,34.0,135.7,106.2,0
1,Burak,R,88,40,"Directions",2,4,56.7,111.0,74.7,1
1,Fatma,L,74,26,"Book",2,5,56.7,135.7,129.4,0
1,Fatma,L,74,26,"My visits",0,5,11.3,135.7,112.1,0
1,Fatma,L,74,26,"Call",1,5,34.0,135.7,119.0,0
1,Fatma,L,74,26,"Directions",2,4,56.7,111.0,109.3,0
2,Ayşe,R,80,30,"Book",1,1,34.0,37.0,46.5,1
2,Ayşe,R,80,30,"My visits",0,0,11.3,12.3,70.9,1
2,Ayşe,R,80,30,"Call",1,0,34.0,12.3,49.3,1
2,Ayşe,R,80,30,"Directions",2,0,56.7,12.3,29.3,1
2,Mehmet,R,90,28,"Book",1,1,34.0,37.0,46.9,1
2,Mehmet,R,90,28,"My visits",0,0,11.3,12.3,70.5,1
2,Mehmet,R,90,28,"Call",1,0,34.0,12.3,48.6,1
2,Mehmet,R,90,28,"Directions",2,0,56.7,12.3,28.1,0
2,Elif,L,81,32,"Book",1,1,34.0,37.0,46.3,1
2,Elif,L,81,32,"My visits",0,0,11.3,12.3,30.5,1
2,Elif,L,81,32,"Call",1,0,34.0,12.3,50.0,1
2,Elif,L,81,32,"Directions",2,0,56.7,12.3,71.5,1
2,Hasan,R,78,22,"Book",1,1,34.0,37.0,48.4,1
2,Hasan,R,78,22,"My visits",0,0,11.3,12.3,69.4,1
2,Hasan,R,78,22,"Call",1,0,34.0,12.3,47.0,1
2,Hasan,R,78,22,"Directions",2,0,56.7,12.3,25.2,1
2,Zeynep,L,76,30,"Book",1,1,34.0,37.0,46.5,1
2,Zeynep,L,76,30,"My visits",0,0,11.3,12.3,29.3,1
2,Zeynep,L,76,30,"Call",1,0,34.0,12.3,49.3,1
2,Zeynep,L,76,30,"Directions",2,0,56.7,12.3,70.9,1
2,Can,L,84,34,"Book",1,1,34.0,37.0,46.1,1
2,Can,L,84,34,"My visits",0,0,11.3,12.3,31.8,1
2,Can,L,84,34,"Call",1,0,34.0,12.3,50.9,1
2,Can,L,84,34,"Directions",2,0,56.7,12.3,72.0,1
2,Selin,R,79,30,"Book",1,1,34.0,37.0,46.5,1
2,Selin,R,79,30,"My visits",0,0,11.3,12.3,70.9,1
2,Selin,R,79,30,"Call",1,0,34.0,12.3,49.3,1
2,Selin,R,79,30,"Directions",2,0,56.7,12.3,29.3,1
2,Emine,R,72,36,"Book",1,1,34.0,37.0,46.0,1
2,Emine,R,72,36,"My visits",0,0,11.3,12.3,72.7,0
2,Emine,R,72,36,"Call",1,0,34.0,12.3,51.7,1
2,Emine,R,72,36,"Directions",2,0,56.7,12.3,33.2,1
2,Burak,R,88,40,"Book",1,1,34.0,37.0,46.1,1
2,Burak,R,88,40,"My visits",0,0,11.3,12.3,74.1,1
2,Burak,R,88,40,"Call",1,0,34.0,12.3,53.7,1
2,Burak,R,88,40,"Directions",2,0,56.7,12.3,36.2,1
2,Fatma,L,74,26,"Book",1,1,34.0,37.0,47.3,1
2,Fatma,L,74,26,"My visits",0,0,11.3,12.3,27.0,1
2,Fatma,L,74,26,"Call",1,0,34.0,12.3,48.0,1
2,Fatma,L,74,26,"Directions",2,0,56.7,12.3,70.1,1"""
src = MY_LAYOUT.strip() or DEFAULT_LAYOUT.strip()
print("Data:", "your own layout from the interactive lab" if MY_LAYOUT.strip()
      else "the results of a typical student")
lay = pd.read_csv(io.StringIO(src))
for rnd, part in lay.groupby("round"):
    cells = part.drop_duplicates("action").set_index("action")
    layout = {a: (int(cells.loc[a, "col"]), int(cells.loc[a, "row"]))
              for a in ACTIONS}
    print(f"round {rnd}: {part.reached.sum()} of {len(part)} taps in reach for "
          f"the ten patients | all patients: {reach_share(layout):.3f}")
    worst = part.groupby("action").reached.mean().sort_values()
    print("   hardest action:", worst.index[0], f"({worst.iloc[0]:.1f})")

**Python note.** `drop_duplicates("action")` keeps one row per action, because every patient reports the same layout. The ten patients are a small sample, so `reach_share` applies your layout to all patients of the clinic.

**Explore.** Move the sliders: The grid shows the cells within reach of one thumb, with the top row first. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(L=80, g=30, hand="R"):
    for r in range(5, -1, -1):
        print(" ".join("[x]" if in_reach(CELL_X[c], CELL_Y[r], hand, L, g)
                       else "[ ]" for c in range(3)))

widgets.interact(explore, L=widgets.IntSlider(value=80, min=60, max=100),
                 g=widgets.IntSlider(value=30, min=10, max=60),
                 hand=widgets.Dropdown(options=["R", "L"], value="R"));

**Observe and reflect.** Which cells does every thumb reach, whatever the hand? Why might the corner right next to the base of the thumb be as hard to reach as the far corner?

## 2. Gestures: The \$1 recognizer &#91;[5](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-5)&#93;

*Lecture: [Gestures: Recognising a stroke](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#gestures)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate_hands" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A gesture is a stroke of the finger that the app turns into a command. Python builds the \$1 recognizer of Wobbrock, Wilson and Li step by step, tests it on strokes drawn with noise and then recognises the strokes that you drew in Part C.

**From the formulas to the code.** `resample(points, 64)` places $N = 64$ points at the interval $I = L/(N - 1)$. `indicative_angle` returns $\omega = \operatorname{atan2}(\bar{y} - y_1, \bar{x} - x_1)$, and `rotate_by` turns the stroke by $-\omega$. `scale_to` stretches it to a square of 250, and `translate_to` moves the centroid to the origin. `path_distance` gives $d = \frac{1}{N}\sum_k \lVert C_k - T_k \rVert$, and the score is $1 - d/(\tfrac{1}{2}\sqrt{250^2 + 250^2})$.

In [ ]:
N_POINTS, SQUARE = 64, 250.0

def resample(points, n=N_POINTS):
    """n points at equal distances along a stroke."""
    p = np.asarray(points, dtype=float)
    seg = np.hypot(*np.diff(p, axis=0).T)
    dist = np.concatenate([[0], np.cumsum(seg)])
    t = np.linspace(0, dist[-1], n)
    return np.column_stack([np.interp(t, dist, p[:, 0]),
                            np.interp(t, dist, p[:, 1])])

def indicative_angle(p):
    """Angle from the centroid to the first point."""
    c = p.mean(axis=0)
    return math.atan2(c[1] - p[0, 1], c[0] - p[0, 0])

def rotate_by(p, angle):
    """The stroke turned by angle (radians) around its centroid."""
    c = p.mean(axis=0)
    R = np.array([[math.cos(angle), -math.sin(angle)],
                  [math.sin(angle), math.cos(angle)]])
    return (p - c) @ R.T + c

def scale_to(p, size=SQUARE):
    """The stroke stretched to a square of the given size."""
    return p * (size / np.maximum(p.max(axis=0) - p.min(axis=0), 1e-6))

def translate_to(p):
    """The stroke moved so that its centroid lies at the origin."""
    return p - p.mean(axis=0)

def normalise(points):
    """The four steps of the $1 recognizer."""
    p = resample(points)
    p = rotate_by(p, -indicative_angle(p))
    return translate_to(scale_to(p))

mark = np.array([[20, 60], [40, 80], [100, 20]], dtype=float)   # a check mark
seg = np.hypot(*np.diff(mark, axis=0).T)
print("segments:", seg.round(3), "| L =", round(seg.sum(), 3),
      "| I =", round(seg.sum() / (N_POINTS - 1), 3))
print("scale factors:", (SQUARE / (mark.max(axis=0) - mark.min(axis=0))).round(3))
print("indicative angle of the resampled check:",
      round(math.degrees(indicative_angle(resample(mark))), 1), "degrees")

**Python note.** `np.hypot(*np.diff(p, axis=0).T)` unpacks the two columns of the steps into the two arguments of `np.hypot`. `scale_to` divides by the width and the height of the bounding box separately, so a stroke drawn small or large becomes the same size.

In [ ]:
TEMPLATES = {   # one stroke per gesture, x to the right and y downwards
    "check":
        [(10, 55), (38, 85), (90, 15)],
    "caret":
        [(10, 85), (50, 15), (90, 85)],
    "zigzag":
        [(10, 25), (30, 75), (50, 25), (70, 75), (90, 25)],
    "arrow":
        [(10, 50), (88, 50), (66, 30), (88, 50), (66, 70)],
    "circle":
        [(50.0, 12.0), (40.2, 13.3), (31.0, 17.1), (23.1, 23.1), (17.1, 31.0),
         (13.3, 40.2), (12.0, 50.0), (13.3, 59.8), (17.1, 69.0), (23.1, 76.9),
         (31.0, 82.9), (40.2, 86.7), (50.0, 88.0), (59.8, 86.7), (69.0, 82.9),
         (76.9, 76.9), (82.9, 69.0), (86.7, 59.8), (88.0, 50.0), (86.7, 40.2),
         (82.9, 31.0), (76.9, 23.1), (69.0, 17.1), (59.8, 13.3), (50.0, 12.0)]}
PHI = 0.5 * (-1 + math.sqrt(5))          # the golden ratio
HALF = 0.5 * math.sqrt(SQUARE ** 2 + SQUARE ** 2)

def path_distance(a, b):
    """Mean distance between corresponding points of two strokes."""
    return float(np.mean(np.hypot(*(a - b).T)))

def distance_at_best_angle(p, t, lo=-45, hi=45, eps=2):
    """Smallest path distance over rotations, by golden section search."""
    lo, hi, eps = math.radians(lo), math.radians(hi), math.radians(eps)
    x1 = PHI * lo + (1 - PHI) * hi
    f1 = path_distance(rotate_by(p, x1), t)
    x2 = (1 - PHI) * lo + PHI * hi
    f2 = path_distance(rotate_by(p, x2), t)
    while abs(hi - lo) > eps:
        if f1 < f2:
            hi, x2, f2 = x2, x1, f1
            x1 = PHI * lo + (1 - PHI) * hi
            f1 = path_distance(rotate_by(p, x1), t)
        else:
            lo, x1, f1 = x1, x2, f2
            x2 = (1 - PHI) * lo + PHI * hi
            f2 = path_distance(rotate_by(p, x2), t)
    return min(f1, f2)

NORMAL = {name: normalise(pts) for name, pts in TEMPLATES.items()}

def recognise(points, threshold=0.8):
    """Best template and its score; no match below the threshold."""
    p = normalise(points)
    d = {name: distance_at_best_angle(p, t) for name, t in NORMAL.items()}
    best = min(d, key=d.get)
    score = 1 - d[best] / HALF
    return (best if score >= threshold else "no match"), score

for name, pts in TEMPLATES.items():
    fwd, back = recognise(pts), recognise(pts[::-1])
    print(f"{name}: drawn as stored -> {fwd[0]} {fwd[1]:.3f} | drawn backwards "
          f"-> {back[0]} {back[1]:.3f}")

**Python note.** The search narrows the range of angles from both ends and keeps the side with the smaller distance, as the paper describes. `pts[::-1]` reverses the order of the points, as if the stroke were drawn from its end to its start. `min(d, key=d.get)` returns the name with the smallest distance.

**Exercise B.** After the four steps, a stroke lies at the mean distance $d = 23.5$ from its best template. Compute its score, first by hand and then with `HALF`, and store it in `score`.

In [ ]:
score = None   # your answer

check("Exercise B", score)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
score = 1 - 23.5 / HALF

check("Exercise B", score)

In [ ]:
def draw_by_hand(points, rng, sd, rot, stretch):
    """A template as a hand might draw it: rotated, stretched and jittered."""
    p = resample(points, 32)
    p = rotate_by(p, math.radians(rot)) * np.array(stretch)
    return p + rng.normal(0, sd, p.shape)

rng = np.random.default_rng(RANDOM_STATE)
for sd, rot in [(2, 10), (5, 25), (8, 40)]:
    hits = []
    for name, pts in TEMPLATES.items():
        for k in range(40):
            st = 1 + rng.uniform(-0.3, 0.3, 2)
            s = draw_by_hand(pts, rng, sd, rng.uniform(-rot, rot), st)
            hits.append(recognise(s)[0] == name)
    print(f"noise {sd}, rotation up to {rot} degrees: accuracy "
          f"{np.mean(hits):.3f} over {len(hits)} strokes")

**Python note.** Each simulated stroke is a template with 32 points, turned by a random angle, stretched differently in x and y and moved point by point. The recognizer keeps its accuracy because it removes the rotation, the size and the position before it compares.

In [ ]:
# Figure: the four steps of the $1 recognizer on one stroke
fig, ax = plt.subplots(1, 4, figsize=(11, 2.9))
raw = draw_by_hand(TEMPLATES["check"], np.random.default_rng(7), 2, 20, (1.2, 0.9))
p1 = resample(raw)
p2 = rotate_by(p1, -indicative_angle(p1))
p3 = translate_to(scale_to(p2))
titles = ["drawn", "64 points", "rotated", "scaled and moved"]
for a, p, t in zip(ax, [raw, p1, p2, p3], titles):
    a.plot(p[:, 0], p[:, 1], "o-", ms=2.5, color="#5A4BB5")
    a.plot(*p[0], "o", color="#B53F2E")
    a.plot(*p.mean(axis=0), "+", color="#1E2433", ms=10)
    a.set_title(t, fontsize=9)
    a.invert_yaxis()
    a.set_aspect("equal")
ax[3].plot(NORMAL["check"][:, 0], NORMAL["check"][:, 1], "--", color="#2E8B57")
plt.tight_layout()
plt.show()

**Python note.** The red dot marks the first point and the cross the centroid. After the rotation, the first point lies straight to the left of the centroid, at the indicative angle 0. The dashed line in the last panel is the template. `invert_yaxis` shows y downwards, as on a screen.

**Paste your results.** Copy the results of Part C of the interactive lab and paste them between the triple quotes of `MY_STROKES` below. If you leave it empty, the code uses the strokes of a typical student.

In [ ]:
MY_STROKES = """
"""
DEFAULT_STROKES = """
round,target,ms,points
1,check,892,"26 61;31 67;37 68;38 77;42 80;46 87;50 86;52 94;58 92;63 87;64 81;
73 80;74 74;77 71;79 68;85 63;88 59;93 54;96 51;98 44;101 39;105 35;111 32;
112 27"
1,caret,840,"29 98;37 88;36 85;41 79;46 72;46 62;51 56;56 55;54 51;60 41;62 34;
65 30;71 28;72 36;80 40;79 44;87 51;90 57;92 64;96 71;100 76;104 85;106 88;
109 98"
1,zigzag,822,"29 29;31 37;35 50;38 56;41 65;42 77;49 81;51 73;58 65;62 61;67 49;
72 41;74 41;77 48;84 59;83 71;85 76;86 83;95 84;96 75;101 65;106 56;112 47;
114 40"
1,arrow,751,"27 63;34 65;45 63;51 64;56 63;67 58;73 60;81 60;90 58;98 59;105 57;
110 52;103 53;95 46;91 42;86 38;98 42;96 50;107 56;111 56;106 62;98 67;94 75;
89 77"
1,circle,817,"75 20;61 19;55 21;41 31;36 35;34 48;32 57;34 72;35 81;47 91;51 96;
64 105;71 102;80 102;93 93;101 86;103 80;109 67;108 58;105 44;98 35;93 25;85 20;
71 18"
1,check,988,"30 60;34 65;31 69;40 74;43 79;47 83;51 88;51 93;56 91;62 87;59 81;
68 81;70 72;75 71;81 70;83 63;87 60;93 57;94 48;96 47;105 44;107 38;113 35;
114 31"
1,caret,794,"29 94;36 89;37 84;40 76;41 71;47 63;52 61;54 53;55 45;60 43;65 36;
67 30;68 27;75 37;79 41;81 47;86 51;88 62;93 65;95 71;102 74;105 80;108 90;
109 94"
1,zigzag,948,"37 31;38 38;40 44;42 58;47 68;44 75;49 81;52 77;56 64;60 54;65 47;
71 38;71 37;75 45;78 58;83 67;81 78;87 85;90 82;95 76;99 65;102 54;112 46;
111 36"
1,arrow,1003,"25 65;38 62;45 63;51 61;58 60;66 61;72 61;83 56;88 57;94 58;
100 58;106 57;100 53;97 47;90 43;85 40;89 45;95 48;102 54;111 61;104 64;100 68;
93 76;89 81"
1,circle,942,"70 16;59 18;50 28;43 33;38 41;37 54;32 66;39 75;44 87;49 96;
57 104;66 102;77 101;85 96;96 94;99 82;103 72;103 63;105 50;98 39;95 30;87 23;
80 16;69 19"
2,check,456,"26 52;33 51;35 64;34 62;41 68;45 78;53 86;43 73;56 77;59 74;68 78;
64 69;77 73;77 69;67 66;96 67;89 64;92 69;99 51;105 47;101 54;111 45;117 42;
111 40"
2,caret,297,"14 84;17 76;28 62;24 66;29 66;38 62;50 52;53 48;58 34;62 30;69 23;
82 23;75 33;87 34;83 36;93 56;96 60;90 77;94 74;102 86;104 95;106 95;105 108;
104 123"
2,zigzag,354,"41 34;42 34;45 60;55 63;49 72;52 79;59 75;52 69;53 66;67 55;64 48;
62 43;68 50;81 40;77 55;78 74;87 77;84 83;90 78;94 69;90 64;102 62;99 36;98 43"
2,arrow,315,"18 52;25 62;40 59;40 59;60 60;63 59;73 68;91 57;96 64;99 66;106 70;
116 71;107 56;102 58;103 41;101 43;101 52;112 52;113 55;117 59;113 66;98 75;
93 83;97 75"
2,circle,421,"52 25;48 29;36 29;34 49;34 59;26 65;28 76;45 83;45 89;55 96;67 97;
80 95;90 99;99 85;102 78;106 73;107 60;105 54;107 40;98 29;86 23;89 29;74 33;
57 21"
2,check,490,"38 57;34 63;40 72;36 77;40 84;51 90;50 95;52 100;54 99;58 94;60 97;
62 84;73 80;75 71;72 66;75 66;86 59;91 56;100 46;97 40;103 34;109 35;109 26;
116 19"
2,caret,461,"45 129;31 99;45 98;47 92;42 78;49 76;55 63;46 62;51 54;47 39;53 29;
60 24;55 29;71 23;76 33;76 50;91 43;94 47;97 55;107 62;104 70;115 74;127 83;
135 83"
2,zigzag,521,"35 35;28 46;38 53;38 51;44 60;35 65;43 61;51 70;46 55;65 54;68 57;
73 42;80 51;77 52;77 61;81 64;75 71;90 75;90 78;95 67;92 62;108 49;104 56;
112 51"
2,arrow,505,"26 59;23 57;41 61;54 63;59 51;68 65;63 59;74 56;85 67;99 56;110 58;
107 56;106 50;100 42;95 43;96 36;95 40;110 48;108 55;113 60;106 64;102 67;94 78;
78 85"
2,circle,465,"79 15;58 25;61 17;53 34;54 38;41 42;41 66;41 73;39 93;47 86;52 97;
60 103;72 109;82 103;86 98;82 83;90 82;95 72;100 60;108 44;86 28;88 25;87 24;
68 27"
"""
src = MY_STROKES.strip() or DEFAULT_STROKES.strip()
print("Data:", "your own strokes from the interactive lab" if MY_STROKES.strip()
      else "the strokes of a typical student")
strokes = pd.read_csv(io.StringIO(src))
strokes["xy"] = [np.array([[float(v) for v in pair.split()]
                           for pair in s.split(";")]) for s in strokes.points]
res = [recognise(p) for p in strokes.xy]
strokes["name"] = [r[0] for r in res]
strokes["score"] = [round(r[1], 3) for r in res]
strokes["right"] = strokes.name == strokes.target
print(strokes.groupby("round")[["right", "score", "ms"]].mean().round(3))
print(strokes.loc[~strokes.right, ["round", "target", "name", "score"]])

**Python note.** Each row holds one stroke as text, pairs of x and y separated by semicolons. The list comprehension turns the text into an array of points, and `groupby("round")` compares the comfortable strokes of round 1 with the fast strokes of round 2.

**Observe and reflect.** Did your fast strokes of round 2 score lower than those of round 1? Which gesture would you draw backwards by mistake, and what would a second template for it change?

## 3. Typing on glass: Speed and errors &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-6)&#93;

*Lecture: [Typing on glass: Text entry speed and errors](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#text-entry)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "recognise" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Text entry is measured by its speed and its errors. Python computes the words per minute, the minimum string distance between the presented and the transcribed text, the error rate and the keystrokes per character. It then compares your phrases of Part D, typed sitting and walking.

**From the formulas to the code.** `wpm(T, s)` computes $\frac{|T| - 1}{s} \times 60 \times \frac{1}{5}$. `msd(a, b)` is the smallest number of insertions, deletions and substitutions that turn $a$ into $b$, `error_rate(P, T)` divides it by $\max(|P|, |T|)$, and `kspc(keys, T)` divides the keystrokes by $|T|$.

In [ ]:
def msd(a, b):
    """Minimum string distance: the fewest edits that turn a into b."""
    D = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    D[:, 0] = np.arange(len(a) + 1)
    D[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1,         # deletion
                          D[i, j - 1] + 1,         # insertion
                          D[i - 1, j - 1] + cost)  # substitution or match
    return int(D[-1, -1])

def wpm(T, seconds):
    """Words per minute: a word is five characters."""
    return (len(T) - 1) / seconds * 60 / 5

def error_rate(P, T):
    """Share of characters in error, from the minimum string distance."""
    return msd(P, T) / max(len(P), len(T))

def kspc(keys, T):
    """Keystrokes per character of the transcribed text."""
    return keys / len(T)

print("wpm:", round(wpm("please call me after five", 9.6), 1))
P, T = "my appointment is on monday", "my appointmnet is on mondy"
print("msd:", msd(P, T), "| error rate:", round(error_rate(P, T), 3),
      "| kspc:", round(kspc(30, T), 3))

**Python note.** `D[i, j]` holds the distance between the first `i` characters of `a` and the first `j` characters of `b`. Each cell takes the cheapest of three moves from its neighbours, so the last cell holds the distance of the whole strings. The time is measured from the first keystroke, so the first character does not count.

**Exercise C.** A patient types a phrase of 26 characters in 9.4 s. Compute the words per minute and store them in `wpm_c`.

In [ ]:
wpm_c = None   # your answer

check("Exercise C", wpm_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
wpm_c = (26 - 1) / 9.4 * 60 / 5

check("Exercise C", wpm_c)

**Paste your results.** Copy the results of Part D of the interactive lab and paste them between the triple quotes of `MY_TYPING` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_TYPING = """
"""
DEFAULT_TYPING = """
round,presented,transcribed,seconds,keystrokes
1,"my appointment is on monday","my appointment is on monday",14.2,27
1,"please call me after five","please call me after five",12.1,27
1,"i need a new prescription","i need a new prescriptoin",13.4,25
1,"the pain started last week","the pain started last week",12.8,28
1,"can i come earlier tomorrow","can i come earlier tomorrow",13.9,27
2,"my test results are ready","my test resulys are ready",15.6,27
2,"i will be ten minutes late","i will be ten minutes lare",16.9,28
2,"where is the blood test room","where is the bloof test room",18.2,30
2,"please send me a reminder","please sebd me a reminder",16.1,27
2,"my daughter will come with me","my daughter will cone with me",18.8,31"""
src = MY_TYPING.strip() or DEFAULT_TYPING.strip()
print("Data:", "your own phrases from the interactive lab" if MY_TYPING.strip()
      else "the phrases of a typical student")
typing = pd.read_csv(io.StringIO(src), keep_default_na=False)
typing["wpm"] = [wpm(t, s) for t, s in zip(typing.transcribed, typing.seconds)]
typing["msd"] = [msd(p, t) for p, t in zip(typing.presented, typing.transcribed)]
typing["error"] = [error_rate(p, t)
                   for p, t in zip(typing.presented, typing.transcribed)]
typing["kspc"] = [kspc(k, t) for k, t in zip(typing.keystrokes, typing.transcribed)]
summary = typing.groupby("round")[["wpm", "error", "kspc"]].mean().round(3)
summary.index = ["sitting", "walking"][:len(summary)]
print(summary)

**Python note.** `keep_default_na=False` keeps an empty transcription as an empty string instead of a missing value. The mean over the five phrases of a round compares sitting with walking.

In [ ]:
# Figure: speed and errors while sitting and while walking
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
summary["wpm"].plot(kind="bar", ax=ax[0], color=["#5A4BB5", "#C98510"], rot=0)
ax[0].axhline(36.2, ls="--", color="#1E2433", label="phones, 36.2 WPM")
ax[0].set_ylabel("words per minute")
ax[0].legend(fontsize=8)
(summary["error"] * 100).plot(kind="bar", ax=ax[1], color=["#5A4BB5", "#C98510"],
                              rot=0)
ax[1].set_ylabel("error rate (percent)")
plt.tight_layout()
plt.show()

**Python note.** The dashed line marks the mean speed of the 37,370 volunteers of Palin and colleagues. With a mouse on a drawn keyboard, your speed says more about the walking than about phones.

**Observe and reflect.** How much did walking cost you in speed and in errors? Which keys did you hit by mistake, and were they neighbours of the right ones?

## 4. Speech: The word error rate &#91;[12](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-12)&#93;

*Lecture: [Talking to the app: Speech and multimodal input](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#speech-and-multimodal)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "kspc" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A speech recognizer turns speech into words, and its errors are counted in words. Python computes the word error rate (WER) with the same distance as for characters. It estimates how often a whole command is recognised and how much reading the booking back helps.

**From the formulas to the code.** `wer(ref, hyp)` computes $WER = (S + D + I)/N$ with the minimum string distance over words. `p_command(w, n)` gives $(1 - w)^n$, the chance that all $n$ words are right if errors are independent. With a read-back that catches a share $c$ of the errors, a wrong booking remains with the probability $(1 - P)(1 - c)$.

In [ ]:
def wer(ref, hyp):
    """Word error rate: word edits divided by the words of the reference."""
    r, h = ref.split(), hyp.split()
    return msd(r, h) / len(r)

def p_command(w, n):
    """Chance that all n words of a command are recognised."""
    return (1 - w) ** n

COMMAND = 'book cardiology on wednesday at ten thirty'
for hyp in ["book cardiology on wednesday at two thirty",
            "look cardiology wednesday at ten thirty"]:
    print(f"{hyp}: WER {wer(COMMAND, hyp):.3f}")
N_WORDS = len(COMMAND.split())
WER_STUDY = {'white speakers': 0.19, 'Black speakers': 0.35}
WER_COMMANDS = {'typical speaker': 0.05,
                'older speaker or strong accent': 0.1}
CATCH = 0.9
for who, w in {**WER_STUDY, **WER_COMMANDS}.items():
    p = p_command(w, N_WORDS)
    print(f"{who}, WER {w}: all {N_WORDS} words right {p:.3f} | wrong bookings "
          f"without read-back {1 - p:.3f}, "
          f"with read-back {(1 - p) * (1 - CATCH):.3f}")

**Python note.** `msd` works on lists as well as on strings, because it only compares elements and counts them. `{**a, **b}` merges two dictionaries into one. The rates of Koenecke and colleagues come from long interviews, while the short commands of an app are easier to recognise.

**Exercise D.** The recognizer hears *book cardiology wednesday at two thirty* for the command *book cardiology on wednesday at ten thirty*. Count the substitutions and deletions by hand, then store the word error rate in `w_err`.

In [ ]:
w_err = None   # your answer

check("Exercise D", w_err)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w_err = wer(COMMAND, "book cardiology wednesday at two thirty")

check("Exercise D", w_err)

In [ ]:
# Figure: whole commands recognised, and wrong bookings with a read-back
words = np.arange(1, 13)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
for (who, w), col in zip({**WER_COMMANDS, **WER_STUDY}.items(),
                         ["#2E8B57", "#0E7A78", "#C98510", "#B53F2E"]):
    ax[0].plot(words, p_command(w, words), "o-", ms=3, color=col,
               label=f"{who}, {w}")
ax[0].axvline(N_WORDS, ls=":", color="#585E6E")
ax[0].set_xlabel("words in the command")
ax[0].set_ylabel("all words right")
ax[0].legend(fontsize=7)
ws = np.linspace(0, 0.4, 41)
ax[1].plot(ws, 1 - p_command(ws, N_WORDS), color="#B53F2E",
           label="without read-back")
ax[1].plot(ws, (1 - p_command(ws, N_WORDS)) * (1 - CATCH), color="#2E8B57",
           label="with read-back")
ax[1].set_xlabel("word error rate")
ax[1].set_ylabel("wrong bookings")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The dotted line marks the seven words of the booking command. Long commands fail more often, which is one reason why voice interfaces confirm what they heard.

**Observe and reflect.** The read-back costs a few seconds every time. For which commands of the clinic app is it worth the time, and for which is it not?

## 5. Notifications and interruptions &#91;[17](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-17)&#93;

*Lecture: [Notifications, interruptions and context](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#notifications)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "p_command" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Notifications arrive at random moments and pull the eyes to the phone. Python models their arrivals as a Poisson process, computes the chance that a task stays free of interruptions and compares delivery at once with delivery in batches. It then replays the waiting room of the 3D scene.

**From the formulas to the code.** With $\lambda$ notifications per hour, a task of $T$ hours stays free of them with the probability `p_quiet(lam, T)` $= e^{-\lambda T}$. Delivered in batches every $B$ hours, a task with $T < B$ is interrupted with the probability `p_batched(lam, T, B)` $= \frac{T}{B}(1 - e^{-\lambda B})$.

In [ ]:
LAM_H = round(63.5 / 16, 2)      # notifications per waking hour
BOOK_H = 3 / 60                    # a booking of three minutes, in hours

def p_quiet(lam, T):
    """Chance that no notification arrives during a task of T hours."""
    return math.exp(-lam * T)

def p_batched(lam, T, B):
    """Chance that a batch delivery falls into the task and is not empty."""
    return T / B * (1 - math.exp(-lam * B))

print("notifications per hour:", LAM_H)
print("booking interrupted, at once:", round(1 - p_quiet(LAM_H, BOOK_H), 3))
print("booking interrupted, batches every hour:",
      round(p_batched(LAM_H, BOOK_H, 1), 4))
rng = np.random.default_rng(RANDOM_STATE)
arrivals = np.cumsum(rng.exponential(1 / LAM_H, 200_000))   # hours
starts = rng.uniform(0, arrivals[-1] - 1, 100_000)          # bookings
nxt = arrivals[np.searchsorted(arrivals, starts)]   # next notification
print("simulated share interrupted:", round(np.mean(nxt < starts + BOOK_H), 3))

**Python note.** In a Poisson process, the number of arrivals in an interval follows the Poisson distribution, and the chance of none is $e^{-\lambda T}$. The simulation draws random arrival times and random bookings and counts how often a notification falls inside a booking.

**Exercise E.** With 3.97 notifications per hour, store the chance that a booking of three minutes stays free of notifications in `p_free`.

In [ ]:
p_free = None   # your answer

check("Exercise E", p_free)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_free = p_quiet(3.97, 3 / 60)

check("Exercise E", p_free)

In [ ]:
LOOK_S = 40        # a notification holds the eyes for 40 s
NAMES = ['Ali', 'Derya', 'Kemal', 'Nur', 'Okan', 'Sevgi', 'Tolga', 'Leyla']

def waiting_room(seed=11, n=8, minutes=40, lam=LAM_H):
    """Browsing, notifications and the call of each patient, in seconds."""
    rng, end, pats = random.Random(seed), minutes * 60, []
    for k in range(n):
        share = round(rng.uniform(0.3, 0.75), 2)
        on, t, browse = rng.random() < share, 0.0, []
        while t < end:
            d = rng.expovariate(1 / (90 if on else 90 * (1 - share) / share))
            if on:
                browse.append((round(t), round(min(t + d, end))))
            t += d
            on = not on
        notes, t = [], rng.expovariate(lam / 3600)
        while t < end:
            notes.append(round(t))
            t += rng.expovariate(lam / 3600)
        pats.append(dict(call=(k + 1) * 300, browse=browse, notes=notes))
    return pats

def looking(p, t, hold):
    """True if the patient looks at the phone at second t."""
    if any(a <= t < b for a, b in p["browse"]):
        return True
    return any(a <= t < a + LOOK_S for a in p["notes"]
               if not (hold and a < p["call"]))

ROOM = waiting_room()
for phone_call in (False, True):
    for hold in (False, True):
        missed = [NAMES[k] for k, p in enumerate(ROOM)
                  if not phone_call and looking(p, p["call"], hold)]
        sent = sum(sum(a < p["call"] for a in p["notes"]) * (not hold)
                   + phone_call for p in ROOM)
        print(f"call to the phone {phone_call!s:5}, others wait {hold!s:5}: "
              f"missed {missed or 'none'}, notifications before the calls {sent}")

**Python note.** `random.Random(seed)` is a generator of the standard library with its own seed, so the waiting room is the same as in the 3D scene. `expovariate` draws the waiting time to the next event of a Poisson process. `!s:5` prints a truth value as text in a field of five characters.

In [ ]:
# Figure: the chance of a quiet task, and the waiting room
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
mins = np.arange(0, 31)
ax[0].plot(mins, [p_quiet(LAM_H, m / 60) for m in mins], color="#B53F2E",
           label="delivered at once")
ax[0].plot(mins, [1 - p_batched(LAM_H, m / 60, 1) for m in mins],
           color="#2E8B57", label="in batches every hour")
ax[0].set_xlabel("length of the task (minutes)")
ax[0].set_ylabel("no interruption")
ax[0].legend(fontsize=8)
for k, p in enumerate(ROOM):
    for a, b in p["browse"]:
        ax[1].plot([a / 60, b / 60], [k, k], color="#C9C3E6", lw=6)
    for a in p["notes"]:
        ax[1].plot([a / 60, (a + LOOK_S) / 60], [k, k], color="#C98510", lw=6)
    miss = looking(p, p["call"], False)
    ax[1].plot(p["call"] / 60, k, "X" if miss else "o", ms=9,
               color="#B53F2E" if miss else "#2E8B57")
ax[1].set_yticks(range(len(NAMES)), NAMES)
ax[1].set_xlabel("minutes in the waiting room")
plt.tight_layout()
plt.show()

**Python note.** Violet bars mark browsing, orange bars the 40 s after a notification. A red cross marks a call that came while the patient looked at the phone.

**Observe and reflect.** Which missed call disappears when other apps wait, and which one remains? What does that tell you about the cause of each miss?

## 6. The clinic app, stage 11 &#91;[20](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#ref-20)&#93;

*Lecture: [The clinic app, stage 11: The app on the phone, with voice](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#the-clinic-app-stage-11)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ROOM" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 11 moves the clinic app onto the phone. Python collects the results of the sections in one dictionary. It holds the reach of the main actions, the time of the reason field and the voice booking. It also holds the calls in the waiting room and the effect of the reminders on attendance.

In [ ]:
STAGE11 = {}
for name, layout in LAYOUTS.items():
    STAGE11[f"one-thumb reach, {name}"] = round(reach_share(layout), 3)
REASON = 60                                # characters of a typical reason
cps = round(36.2 * 5 / 60, 3)              # characters per second
STAGE11["typing the reason (s)"] = round((REASON - 1) / cps, 1)
STAGE11["errors left in the reason"] = round(REASON * 0.023, 2)
STAGE11["choosing the reason (taps)"] = 2
for who, w in WER_COMMANDS.items():
    p = round(p_command(w, N_WORDS), 3)
    STAGE11[f"voice command right, {who}"] = p
    STAGE11[f"wrong bookings with read-back, {who}"] = round(
        (1 - p) * (1 - CATCH), 3)
STAGE11["missed calls in the waiting room, display only"] = sum(
    looking(p, p["call"], False) for p in ROOM)
STAGE11["missed calls, call sent to the phone"] = 0

ATTEND0, OR_SMS = 0.82, 1.48
odds0 = round(ATTEND0 / (1 - ATTEND0), 3)
odds1 = round(odds0 * OR_SMS, 3)
STAGE11["attendance without reminders"] = ATTEND0
STAGE11["attendance with reminders"] = round(odds1 / (1 + odds1), 3)
for lo_hi in (1.23, 1.72):
    o = round(odds0 * lo_hi, 3)
    print(f"odds ratio {lo_hi}: attendance {o / (1 + o):.3f}")
for key, value in STAGE11.items():
    print(f"{key}: {value}")

**Python note.** The odds of attending are the share who attend divided by the share who do not. An odds ratio multiplies the odds, not the share, so the share is computed back from the new odds. Each intermediate value is rounded to three decimals, as in the lecture.

**Exercise F.** Without reminders, 0.82 of the patients attend. Compute the odds, multiply them by the odds ratio 1.48 and store the share who attend with reminders in `p_att`.

In [ ]:
p_att = None   # your answer

check("Exercise F", p_att)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
o = 0.82 / 0.18 * 1.48
p_att = o / (1 + o)

check("Exercise F", p_att)

In [ ]:
# Figure: reach, voice and attendance of stage 11
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
r = [STAGE11["one-thumb reach, current"], STAGE11["one-thumb reach, stage 11"]]
ax[0].bar(["current", "stage 11"], r, color=["#B53F2E", "#2E8B57"])
ax[0].set_ylim(0, 1)
ax[0].set_ylabel("main actions in one-thumb reach")
wb = [1 - STAGE11[f"voice command right, {w}"] for w in WER_COMMANDS]
wr = [STAGE11[f"wrong bookings with read-back, {w}"] for w in WER_COMMANDS]
x = np.arange(len(WER_COMMANDS))
ax[1].bar(x - 0.2, wb, 0.4, color="#B53F2E", label="no read-back")
ax[1].bar(x + 0.2, wr, 0.4, color="#2E8B57", label="read-back")
ax[1].set_xticks(x, ["typical", "older or accent"])
ax[1].set_ylabel("wrong bookings")
ax[1].legend(fontsize=8)
a = [STAGE11["attendance without reminders"], STAGE11["attendance with reminders"]]
ax[2].bar(["no reminder", "reminder"], a, color=["#B53F2E", "#2E8B57"])
ax[2].set_ylim(0.6, 1)
ax[2].set_ylabel("attendance")
plt.tight_layout()
plt.show()

**Python note.** The reminders act through the odds ratio of the randomised trials in the review of Guy and colleagues. The attendance of 0.82 without reminders is an illustrative value for the clinic.

## 7. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE11" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-11/"
               "NB11_mobile_and_multimodal.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The methods of this week apply to any app. Each app below has a main action at a cell of the grid, a text field that users type and a number of notifications per day.

The values are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "food_delivery"
# clinic, banking, e_government, food_delivery, course_registration
APPS = {
    "clinic": ((2, 5), 60, 6),
    "banking": ((1, 0), 34, 9),
    "e_government": ((0, 5), 120, 3),
    "food_delivery": ((1, 0), 45, 14),
    "course_registration": ((2, 4), 25, 5)}

(c, r), chars, per_day = APPS[APP]
print(f"{APP}: main action at column {c}, row {r}: "
      f"{cell_share(c, r):.3f} of users reach it with one thumb")
t = (chars - 1) / round(36.2 * 5 / 60, 3)
print(f"  typing {chars} characters: {t:.1f} s at 36.2 WPM, "
      f"{chars * 0.023:.2f} errors left on average")
lam = per_day / 16
print(f"  {per_day} notifications per day: a task of 3 minutes is "
      f"interrupted with probability {1 - p_quiet(lam, 3 / 60):.3f}")

**Python note.** Each entry holds the cell of the main action, the characters of a typical text field and the notifications per day. The rates of Palin and colleagues stand in for the users of each app.

**Observe and reflect.** Which barrier of your app costs its users the most: The reach, the typing or the interruptions? Which one would you fix first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Add a template for a circle drawn clockwise and measure how the accuracy on backward strokes changes. |
| Web development | Measure where the main buttons of a mobile website lie and compute with `cell_share` how many users reach them with one thumb. |
| Mobile development | Design a layout of five actions that every thumb of `THUMB_PATIENTS` reaches, and check it with `in_reach`. |
| Game development | Use `recognise` for a spell-casting game with five gestures, and set the threshold so that fewer than 2 percent of the fast strokes cast the wrong spell. |
| Embedded systems and the Internet of Things | A smartwatch shows notifications on the wrist. Compute `p_quiet` for a run of 30 minutes with 6 notifications per hour and with batches every 15 minutes. |
| Data science | Simulate 1,000 voice commands with independent word errors and compare the share of fully correct commands with `p_command`. |
| Cybersecurity | A lock screen shows the first 40 characters of each notification. Write a rule that hides health terms and test it on ten messages of the clinic. |
| Health informatics | Estimate the no-shows that reminders avoid in a clinic with 400 appointments a day and an attendance of 0.78, with the odds ratio and its confidence interval. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-11).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>